# Notebook 4 — Thesis Results Visualisation
Reproduce all figures for the thesis from pre-computed results.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

TICKERS = ['AAPL', 'MSFT', 'TSLA', 'NVDA', 'GOOGL']
MODELS  = ['LSTM-Sentiment', 'LSTM-Only', 'ARIMA', 'SVR']
METRICS = ['rmse', 'mae', 'mape', 'directional_accuracy']
COLOURS = {'LSTM-Sentiment': '#10b981', 'LSTM-Only': '#ef4444', 'ARIMA': '#f59e0b', 'SVR': '#8b5cf6'}

## 1. Load metrics

In [ ]:
metrics_path = Path('../results/metrics.json')
if not metrics_path.exists():
    raise FileNotFoundError('Run evaluate_all.py first')
with open(metrics_path) as f:
    metrics_raw = json.load(f)
df_m = pd.DataFrame(metrics_raw)
df_m.head()

## 2. RMSE comparison across tickers

In [ ]:
pivot = df_m.pivot_table(index='ticker', columns='model', values='rmse')
pivot = pivot.reindex(columns=MODELS)

ax = pivot.plot(kind='bar', figsize=(12, 5), color=[COLOURS[m] for m in MODELS], edgecolor='white', width=0.7)
ax.set_title('RMSE by Ticker and Model', fontsize=13)
ax.set_xlabel('Ticker')
ax.set_ylabel('RMSE ($)')
ax.legend(title='Model', bbox_to_anchor=(1, 1))
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/fig_rmse_comparison.png', bbox_inches='tight')
plt.show()

## 3. Directional accuracy heatmap

In [ ]:
pivot_da = df_m.pivot_table(index='model', columns='ticker', values='directional_accuracy')
pivot_da = pivot_da.reindex(index=MODELS, columns=TICKERS)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_da, annot=True, fmt='.1f', cmap='YlGn', vmin=40, vmax=80,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Dir Acc %'})
ax.set_title('Directional Accuracy (%) by Model × Ticker', fontsize=12)
plt.tight_layout()
plt.savefig('../results/fig_dir_acc_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Prediction vs actual — all models, one ticker

In [ ]:
TICKER = 'AAPL'
pred_path = Path(f'../results/predictions/{TICKER}_test_predictions.csv')
if not pred_path.exists():
    raise FileNotFoundError(f'{pred_path} not found — train all models first')
df_p = pd.read_csv(pred_path, parse_dates=['date'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_p['date'], df_p['actual'], label='Actual', color='#1e40af', linewidth=1.5)
col_map = {
    'lstm_sentiment_pred': ('LSTM-Sentiment', COLOURS['LSTM-Sentiment'], (6, 2)),
    'lstm_only_pred':      ('LSTM-Only',      COLOURS['LSTM-Only'],      (3, 3)),
    'arima_pred':          ('ARIMA',          COLOURS['ARIMA'],          (4, 4)),
    'svr_pred':            ('SVR',            COLOURS['SVR'],            (2, 4)),
}
for col, (label, color, dash) in col_map.items():
    if col in df_p.columns:
        ax.plot(df_p['date'], df_p[col], label=label, color=color, linewidth=1.2,
                linestyle=(0, dash))
ax.set_title(f'{TICKER} — Test Set: All Models', fontsize=12)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
plt.tight_layout()
plt.savefig(f'../results/fig_{TICKER}_predictions.png', bbox_inches='tight')
plt.show()

## 5. Full metrics table (LaTeX export for thesis)

In [ ]:
table = df_m.pivot_table(index=['ticker', 'model'], values=METRICS).round(2)
table = table.reindex(pd.MultiIndex.from_product([TICKERS, MODELS]))
print(table.to_latex(bold_rows=True, caption='Model Performance Comparison', label='tab:results'))